## CIS 635: Assignment 2 — Regression with Linear Regression and k-Nearest Neighbors

**Data**: Medical insurance charges (`insurance.csv`).

**Target:** `charges` - individual annual medical costs.

**Features:** `age`, `sex`, `bmi`, `children`, `smoker`, `region`.

**Random seed:** `RANDOM_SEED = 97`, used for the 80/20 train/test split. Every other step is deterministic, so re-running the notebook reproduces every number.

**Evaluation rules followed:** the test set is held out once, before any model fitting or tuning and used only for final evaluation. Anything that learns from the data sits inside a `pipeline` fit on training data only, including inside cross-validation. Every model that uses the features is fit on standardized features. The only special case is the unscaled kNN in task c.i, which the assignment asks for as a comparison. Numerical results are reported to three decimal places.

### Section 0: Setup & Data Loading

This cell imports the libraries we need, sets the random seed and test proportion, and loads the insurance CSV directly from GitHub into a pandas DataFrame. It prints the shape, column names and data types, and shows the first rows so we can see which features are numeric and which are categorical.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

DATA_URL = "https://raw.githubusercontent.com/mdkamrulhasan/data_mining_kdd/main/data/medical-cost/insurance.csv"

RANDOM_SEED = 97
TEST_PROP = 0.2

insurance = pd.read_csv(DATA_URL)

print("\nRandom seed: ", RANDOM_SEED)
print("\ninsurance.csv shape: ", insurance.shape)
print("\nColumns: ", insurance.columns.tolist())
print("\nData types:")
print(insurance.dtypes.to_string())

print()
insurance.head()

The file has 1,338 rows and 7 columns. `age`, `bmi`, `children` and `charges` are numeric; `sex`, `smoker` and `region` are categorical and need to be encoded.

### Section 1: Missing Values & Duplicate Records - Task a.i

This cell counts NULLs per column and in total. It then finds fully duplicated rows where every column, including `charges`, matches another row. Duplicates are dropped, and the cell prints the row count before and after.

In [ ]:
null_counts = insurance.isnull().sum()
total_nulls = null_counts.sum()

print("\nNULLs per column:")
print(null_counts.to_string())
print("\nTotal NULL cells: ", total_nulls)

dup_mask = insurance.duplicated(keep=False)
dup_count = insurance.duplicated().sum()

print("\nDuplicate rows (exact copies of an earlier row): ", dup_count)
print("\nAll rows involved in duplication:")
print(insurance[dup_mask])

rows_before = len(insurance)
insurance = insurance.drop_duplicates().reset_index(drop=True)
rows_after = len(insurance)

print("\nRows before dropping duplicates: ", rows_before)
print("\nRows after dropping duplicates: ", rows_after)
print("\nRemaining duplicate rows: ", insurance.duplicated().sum())

There are no missing values in any column, so no imputation is needed.

There is one exact duplicate: rows 195 and 581 are identical in all seven columns. We keep one copy. Dropping it prevents the same record being in test set twice. Or landing in both test and train sets, where kNN would find it at distance 0 causing a data leakage.

### Section 2: Encoding Categorical Features - Task a.i

This cell lists the categorical features and then encodes them:

- `sex` and `smoker` (two categories each) are **label encoded** into one 0/1 column each: female = 0, male = 1, and no = 0, yes = 1.
- `region` (four categories with no natural order) is **one-hot encoded** into four 0/1 columns, one per region.

It then checks that no value was left unmapped and shows the encoded DataFrame.

In [ ]:
categorical_cols = ["sex", "smoker", "region"]

for col in categorical_cols:
    print("\n" + col + " categories: ", sorted(insurance[col].unique()))

insurance_enc = insurance.copy()

# Label encoding for the two binary features
insurance_enc["sex"] = insurance_enc["sex"].map({"female": 0, "male": 1})
insurance_enc["smoker"] = insurance_enc["smoker"].map({"no": 0, "yes": 1})

# One-hot encoding for the nominal feature with four categories
insurance_enc = pd.get_dummies(insurance_enc, columns=["region"], dtype=int)

unmapped = insurance_enc[["sex", "smoker"]].isnull().sum().sum()

print("\nUnmapped values after label encoding: ", unmapped)
print("\nEncoded shape: ", insurance_enc.shape)
print("\nEncoded columns: ", insurance_enc.columns.tolist())

print()
insurance_enc.head()

**Label encoding for `sex` and `smoker`**: each has only two categories, so a single 0/1 column carries all the information. With only two values there is only one distance between them, so label encoding does not invent a false ordering.

**One-hot encoding for `region`**: the four regions are nominal meaning they have no particular order. Label encoding them as 0, 1, 2, 3 would tell the models that, for example, northeast (0) is closer to northwest (1) than to southwest (3). This can introduce a bias in the vector space, and it would hurt the models. kNN would measure distances on that fake scale, and linear regression would force the region effect to change in equal steps from 0 to 3. One-hot encoding puts every region at the same distance from every other region.

### Section 3: Distribution of Charges - Task a.ii

This cell computes descriptive statistics for `charges` (mean, median, standard deviation, min, max, Q1, Q3, IQR and skewness) and draws a histogram with the mean and median marked, so we can describe the shape of the target.

In [ ]:
charges = insurance_enc["charges"]

q1, q3 = charges.quantile(0.25), charges.quantile(0.75)
charges_stats = {
    "Mean": charges.mean(),
    "Median": charges.median(),
    "Std Dev": charges.std(),
    "Min": charges.min(),
    "Max": charges.max(),
    "Q1": q1,
    "Q3": q3,
    "IQR": q3 - q1,
    "Skewness": charges.skew(),
}

print("\n--- charges (USD) ---")
for name, value in charges_stats.items():
    print(name, ":", round(value, 3))

# print("\nShare of people with charges below 15,000 USD: ", round((charges < 15000).mean(), 3))
# print("\nShare of people with charges above 30,000 USD: ", round((charges > 30000).mean(), 3))

plt.figure(figsize=(10, 5))
plt.hist(charges, bins=40, color="blue", edgecolor="black")
plt.axvline(charges_stats["Mean"], color="red", linestyle="--", linewidth=2,
            label=f"Mean = {charges_stats['Mean']:,.0f} USD")
plt.axvline(charges_stats["Median"], color="orange", linestyle="-", linewidth=2,
            label=f"Median = {charges_stats['Median']:,.0f} USD")
plt.title("Histogram: Annual Medical Charges")
plt.xlabel("Annual medical charges (USD)")
plt.ylabel("Number of people")
plt.legend()
plt.tight_layout()
plt.savefig("charges_histogram.png", dpi=150)
plt.show()

The distribution of `charges` is strongly **right-skewed** (skewness ~ 1.515).

The median is \$9,386.161. The mean (\$13,279.121) is pulled about \$3,900 above the median by a long right tail that reaches \$63,770.428. Besides the main peak at low charges, there is a smaller, flatter hump between \$35,000–\$50,000.

The spread is large (standard deviation \$12,110.360, IQR \$11,911.373).

### Section 4: Charges vs Age and BMI, Colored by Smoker - Task a.ii

This cell plots `charges` against `age` (left) and against `bmi` (right), with non-smokers and smokers in different colors.

Both panels share the same y-axis. It then compares mean and median charges across smoker status and across BMI below vs greater-equal to 30 to show charges on the plot.

In [ ]:
smoker_groups = [
    (0, "Non-smoker", "green"),
    (1, "Smoker", "red")
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for value, label, color in smoker_groups:
    group = insurance_enc[insurance_enc["smoker"] == value]
    axes[0].scatter(group["age"], group["charges"], alpha=0.5, s=15, color=color, label=label)
    axes[1].scatter(group["bmi"], group["charges"], alpha=0.5, s=15, color=color, label=label)

axes[0].set_title("Charges vs Age, colored by Smoker")
axes[0].set_xlabel("Age (years)")
axes[0].set_ylabel("Annual medical charges (USD)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].axvline(30, color="gray", linestyle=":", label="BMI = 30")
axes[1].set_title("Charges vs BMI, colored by Smoker")
axes[1].set_xlabel("BMI (kg/m²)")
axes[1].set_ylabel("Annual medical charges (USD)")
axes[1].tick_params(labelleft=True)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("charges_vs_age_bmi_by_smoker.png", dpi=150)
plt.show()

eda = insurance_enc[["smoker", "bmi", "charges"]].copy()
eda["smoker"] = eda["smoker"].map({0: "Non-smoker", 1: "Smoker"})
eda["bmi_group"] = np.where(eda["bmi"] >= 30, "BMI >= 30", "BMI < 30")

print("\nCharges by smoker status (USD):")
print(eda.groupby("smoker")["charges"].agg(["count", "mean", "median"]).round(3).to_string())

print("\nCharges by smoker status and BMI group (USD):")
print(eda.groupby(["smoker", "bmi_group"])["charges"].agg(["count", "mean", "median"]).round(3).to_string())

share_smokers_high = (eda.loc[eda["charges"] > 30000, "smoker"] == "Smoker").mean()
print("\nShare of people with charges above 30,000 USD who are smokers: ", round(share_smokers_high, 3))

**What the plots show:**

- **Charges vs age**: forms three roughly parallel bands that all rise steadily with age. The bottom band is almost all non-smokers, the top band is almost all smokers, and the middle band mixes both non-smokers and smokers, where non-smokers are the higher cost group of their corresponding group and smokers being low cost group of their corresponding group.
- **Charges vs BMI** shows that BMI moves non-smokers' charges slightly - mean \$7,977.030 below BMI 30 vs \$8,855.531 at or above. But for smokers there is a clear jump at about BMI 30: mean charges go from \$21,363.217 (BMI < 30) to \$41,557.990 (BMI >= 30). So the effect of BMI depends on smoking.
- Smoking is the single biggest driver: smokers average \$32,050.232 vs \$8,440.660 for non-smokers, and 93.8% of people with charges above \$30,000 are smokers.

**What we expect each model to find easy or hard:**

*Linear regression (LR)*:
- *Easy:* the steady, roughly linear rise of `charge` with `age` (the bands share a similar slope), and the large constant gap between smokers and non-smokers, which a single `smoker` coefficient can capture.
- *Hard:* the `smoker` and `BMI` jump. LR uses one BMI slope for everyone, so it will average the flat non-smoker pattern with the steep smoker jump. We expect it to under-predict high BMI smokers and over-predict low BMI smokers. A straight line is also not bounded at 0, so it may predict impossible negative costs for young, healthy people.

*kNN regression*:
- *Easy:* local, non-linear patterns such as the BMI-30 jump for smokers. A high BMI smoker's nearest neighbors are other high BMI smokers of similar age, so no straight-line assumption is needed. As long as the features are scaled so that smoker status actually counts in the distance.
- *Hard:* without scaling, distances are dominated by `age` (range 18–64) and `bmi` (range ~ 16–53), and the 0/1 `smoker` column is almost ignored. Sparse regions, such as older smokers with very high BMI, have few truly close neighbors. This represents curse-of-dimensionality idea - data become sparse as the feature space grows.

*Hard for both:* the non-smokers in the middle band. Nothing in the six features explains why these people cost more, so neither model can predict them well, and the long tail will inflate RMSE for both.

### Section 5: Train/Test Split

This cell separates the features (X) from the target (y = `charges`) and holds out 20% of the rows as a test set using the fixed random seed. From here on, every model is fit and every hyperparameter is tuned on the training set only. The test set is used only to report final metrics. The EDA above was for understanding of the data, but no statistic from it is used in any model.

In [ ]:
X = insurance_enc.drop(columns=["charges"])
y = insurance_enc["charges"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_PROP, random_state=RANDOM_SEED)

print("\nFeatures used: ", X.columns.tolist())
print("\nX_train shape: ", X_train.shape, "| y_train shape: ", y_train.shape)
print("\nX_test shape: ", X_test.shape, "| y_test shape: ", y_test.shape)
print("\nTraining-set mean of charges (USD): ", round(y_train.mean(), 3))
print("\nTest-set mean of charges (USD): ", round(y_test.mean(), 3))
print("\nShare of smokers (train / test): ", round(X_train["smoker"].mean(), 3), "/", round(X_test["smoker"].mean(), 3))

### Section 6: Naive Baseline vs Linear Regression - Task b.i

This cell defines a helper that returns the three test metrics: R^2, MAE and RMSE. It then builds the **naive baseline**, which predicts the training-set mean of `charges` for every test person, and fits a **linear regression** on the training set.

Linear regression is fit inside a `Pipeline` with a `StandardScaler`, so the scaling is learned from the training set only. Scaling does not change a linear model's predictions, and in Task b.ii the coefficients are converted back to dollars per original unit. The training RMSE is also printed as an overfitting check.

In [ ]:
def evaluate(model_name, y_true, y_pred):
    return {
        "Model": model_name,
        "R2": round(r2_score(y_true, y_pred), 3),
        "MAE (USD)": round(mean_absolute_error(y_true, y_pred), 3),
        "RMSE (USD)": round(np.sqrt(mean_squared_error(y_true, y_pred)), 3),
    }


# Naive baseline
baseline_value = y_train.mean()
y_pred_baseline = np.full(len(y_test), baseline_value)

# Linear regression (scaler fit on training data only)
regr_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LinearRegression())
])
regr_lr.fit(X_train, y_train)
y_pred_lr = regr_lr.predict(X_test)

results_b = pd.DataFrame([
    evaluate("Baseline (train mean)", y_test, y_pred_baseline),
    evaluate("Linear Regression", y_test, y_pred_lr),
])

lr_train_rmse = np.sqrt(mean_squared_error(y_train, regr_lr.predict(X_train)))

print("\nBaseline prediction for every test row (USD): ", round(baseline_value, 3))
print("\nTest-set metrics:")
print(results_b.to_string(index=False))
print("\nLinear Regression training RMSE (USD): ", round(lr_train_rmse, 3))

Linear regression is far better than the naive baseline on every test metric:

- **R^2**: 0.738 vs −0.000. The baseline explains none of the variation, so its R^2 is about 0 by construction.
- **MAE:** \$4267.839 vs \$9003.455, so the typical error drops by about 53%.
- **RMSE:** \$6238.145 vs \$12179.404, about 49% lower.

RMSE is clearly larger than MAE for linear regression, which signals some large individual errors. RMSE squares them, so a few big misses weigh a lot. The training RMSE (\$5998.079) is close to the test RMSE, so the model is not overfitting.

#### Visualizing the Baseline vs Linear Regression Comparison - Task b.i

This cell turns the comparison above into three panels:

1. **Test-set MAE and RMSE** for the baseline and linear regression side by side, with each model's R^2 under its name.
2. **Actual vs predicted charges** on the test set. Points on the dashed y = x line are perfect predictions. The baseline appears as a flat line because it predicts the same value for everyone.
3. **Training vs test RMSE** for both models, as in the class notebooks, to check for overfitting.

It also prints the training RMSE of both models and the share of test predictions that land within \$5,000 of the actual charge.

In [ ]:
# Training-set RMSE of the baseline
baseline_train_rmse = np.sqrt(mean_squared_error(y_train, np.full(len(y_train), baseline_value)))

model_names = results_b["Model"].tolist()
x_pos = np.arange(len(model_names))
bar_width = 0.35

fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))

# (1) Test-set MAE and RMSE, with R^2 shown under each model name
mae_bars = axes[0].bar(x_pos - bar_width / 2, results_b["MAE (USD)"], bar_width,
                       color="blue", edgecolor="black", label="MAE")
rmse_bars = axes[0].bar(x_pos + bar_width / 2, results_b["RMSE (USD)"], bar_width,
                        color="orange", edgecolor="black", label="RMSE")
axes[0].bar_label(mae_bars, fmt="{:,.0f}", padding=3)
axes[0].bar_label(rmse_bars, fmt="{:,.0f}", padding=3)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels([f"{name}\nR² = {r2:.3f}" for name, r2 in zip(model_names, results_b["R2"])])
axes[0].set_title("Test-set Errors: Baseline vs Linear Regression")
axes[0].set_ylabel("Error (USD)")
axes[0].set_ylim(0, results_b["RMSE (USD)"].max() * 1.15)
axes[0].legend()
axes[0].grid(True, axis="y", alpha=0.3)

# (2) Actual vs predicted charges on the test set
low = min(0, y_pred_lr.min()) - 2000
high = max(y_test.max(), y_pred_lr.max()) + 2000

axes[1].scatter(y_test, y_pred_baseline, s=15, alpha=0.6, color="gray", label="Baseline (train mean)")
axes[1].scatter(y_test, y_pred_lr, s=15, alpha=0.6, color="purple", label="Linear Regression")
axes[1].plot([low, high], [low, high], color="black", linestyle="--", linewidth=1,
             label="Perfect prediction (y = x)")
axes[1].set_xlim(low, high)
axes[1].set_ylim(low, high)
axes[1].set_title("Actual vs Predicted Charges (test set)")
axes[1].set_xlabel("Actual charges (USD)")
axes[1].set_ylabel("Predicted charges (USD)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# (3) Training vs test RMSE
train_rmse = [baseline_train_rmse, lr_train_rmse]
test_rmse = results_b["RMSE (USD)"].tolist()

train_bars = axes[2].bar(x_pos - bar_width / 2, train_rmse, bar_width,
                         color="teal", edgecolor="black", label="Training RMSE")
test_bars = axes[2].bar(x_pos + bar_width / 2, test_rmse, bar_width,
                        color="darkorange", edgecolor="black", label="Test RMSE")
axes[2].bar_label(train_bars, fmt="{:,.0f}", padding=3)
axes[2].bar_label(test_bars, fmt="{:,.0f}", padding=3)
axes[2].set_xticks(x_pos)
axes[2].set_xticklabels(model_names)
axes[2].set_title("Training vs Test RMSE")
axes[2].set_ylabel("RMSE (USD)")
axes[2].set_ylim(0, max(train_rmse + test_rmse) * 1.15)
axes[2].legend()
axes[2].grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("baseline_vs_linear_regression.png", dpi=250)
plt.show()

within_5k_baseline = (np.abs(y_test.values - y_pred_baseline) <= 5000).mean()
within_5k_lr = (np.abs(y_test.values - y_pred_lr) <= 5000).mean()

print("\nBaseline training RMSE (USD): ", round(baseline_train_rmse, 3))
print("\nLinear Regression training RMSE (USD): ", round(lr_train_rmse, 3))
print("\nShare of test predictions within 5,000 USD of the actual charge:")
print("  Baseline: ", round(within_5k_baseline, 3), "| Linear Regression: ", round(within_5k_lr, 3))

**What the plots show:**

- **Left:** linear regression cuts MAE by more than half and RMSE by about half (MAE \$9,003.455 -> \$4,267.839; RMSE \$12,179.404 -> \$6,238.145), and R^2 rises from −0.000 to 0.738.
- **Middle:** the baseline is a flat line at \$13,275.092. Everyone gets the same prediction, whether their actual charge was \$1,000 or \$60,000.
    - Linear regression's points follow the y = x line closely for charges under about \$15,000. Overall, 69.4% of its test predictions are within \$5,000 of the actual charge, vs 35.4% for the baseline.
    - A group of points sits well below the line at actual charges of about \$11,000–\$30,000: expensive people predicted to be cheap.
    - Two predictions are even below \$0.
- **Right:** linear regression's training RMSE (\$5,998.079) and test RMSE (\$6,238.145) are close, so it is not overfitting. The baseline's training and test RMSE are also close (\$12,087.315 vs \$12,179.404) because the training and test sets have the same share of smokers (20.5%), so their charges are similarly spread out.

### Section 7: Interpreting the Coefficients - Task b.ii

This cell prints the intercept and the coefficient of every feature, then pulls out `smoker` and `bmi`. Each coefficient is the change in _predicted annual charges (USD)_ for a one-unit increase in that feature, holding all other features fixed.

The model was fit on standardized features, so its raw coefficients are in USD per one standard deviation. Dividing by the training standard deviation stored in the scaler (`scale_`) converts them back to USD per original unit. The intercept is adjusted the same way, using the training means stored in the scaler (`mean_`).

In [ ]:
lr_model = regr_lr.named_steps["lr"]
lr_scaler = regr_lr.named_steps["scaler"]

# Convert from USD per 1 std back to USD per original unit
coef_lr = pd.Series(lr_model.coef_ / lr_scaler.scale_, index=X_train.columns)
intercept_lr = lr_model.intercept_ - (coef_lr * lr_scaler.mean_).sum()

print("\nIntercept (USD): ", round(intercept_lr, 3))
print("\nCoefficients (USD per one-unit increase in the feature):")
print(coef_lr.round(3).to_string())

smoker_coef = coef_lr["smoker"]
bmi_coef = coef_lr["bmi"]

print("\nsmoker coefficient: ", round(smoker_coef, 3), "USD per year (smoker = 1 vs non-smoker = 0)")
print("\nbmi coefficient: ", round(bmi_coef, 3), "USD per year per 1 kg/m^2 of BMI")
print("\nPredicted extra cost of a 10-point higher BMI (USD per year): ", round(10 * bmi_coef, 3))

**`smoker` = 23,898.909 USD per year.** Holding age, sex, BMI, number of children and region fixed, the model predicts that a smoker's annual medical charges are about **\$23,899 higher** than those of an otherwise identical non-smoker.

**`bmi` = 334.071 USD per year per kg/m^2.** Holding the other features fixed, each 1-point increase in BMI (1 kg/m²) adds about **\$334 per year** to annual medical charges. For example, comparing BMI 25 with BMI 35 adds about \$3,341 per year.

**Notes**:

- The BMI coefficient is one average slope for everyone. Section 4 showed BMI matters mainly for smokers (the jump at BMI 30), so \$334 understates the BMI effect for smokers and overstates it for non-smokers.
- The four one-hot region columns always add up to 1, so together they overlap with the intercept. This means the individual region coefficients are not uniquely determined and should not be read on their own.

### Section 8: kNN with k = 5, Unscaled vs Standardized - Task c.i

This cell fits the kNN regressor with sklearn's default k = 5 twice on the same training data:

1. on the raw encoded features, and
2. inside a `Pipeline` with a `StandardScaler`, so each feature is rescaled to mean 0 and standard deviation 1 using training-set statistics only.

Both versions are evaluated on the test set. The unscaled version is fit only for this comparison and is not used in any later section.

In [ ]:
# (1) kNN on unscaled features
knn_raw = KNeighborsRegressor(n_neighbors=5)
knn_raw.fit(X_train, y_train)
y_pred_knn_raw = knn_raw.predict(X_test)

# (2) kNN on standardized features (scaler fit on training data only)
knn_scaled = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsRegressor(n_neighbors=5))
])
knn_scaled.fit(X_train, y_train)
y_pred_knn_scaled = knn_scaled.predict(X_test)

results_c1 = pd.DataFrame([
    evaluate("kNN k=5 (unscaled)", y_test, y_pred_knn_raw),
    evaluate("kNN k=5 (standardized)", y_test, y_pred_knn_scaled),
])

print("\nTest-set metrics:")
print(results_c1.to_string(index=False))

To explain the gap, this cell (1) prints the standard deviation of each training feature, which shows how much each feature can contribute to a Euclidean distance before scaling. (2) looks inside the predictions with `kneighbors()` - for every test person it finds the 5 nearest training neighbors and measures how often those neighbors share the test person's smoker status.

In [ ]:
print("\nStandard deviation of each feature (training set, unscaled):")
print(X_train.std().round(3).to_string())

# 5 nearest training neighbors of each test person, unscaled
_, idx_raw = knn_raw.kneighbors(X_test)

# 5 nearest training neighbors of each test person, standardized
# (the test set is transformed with the scaler that was fit on the training set)
scaler_fit = knn_scaled.named_steps["scaler"]
X_test_scaled = scaler_fit.transform(X_test)
_, idx_scaled = knn_scaled.named_steps["knn"].kneighbors(X_test_scaled)

smoker_train = X_train["smoker"].values
smoker_test = X_test["smoker"].values
is_smoker = smoker_test == 1

match_raw = smoker_train[idx_raw] == smoker_test[:, None]
match_scaled = smoker_train[idx_scaled] == smoker_test[:, None]

print("\nShare of the 5 neighbors with the same smoker status (all test rows):")
print("  unscaled: ", round(match_raw.mean(), 3), "| standardized: ", round(match_scaled.mean(), 3))

print("\nShare of the 5 neighbors who are also smokers (test smokers only):")
print("  unscaled: ", round(match_raw[is_smoker].mean(), 3), "| standardized: ", round(match_scaled[is_smoker].mean(), 3))

smoker_in_age_units = (1 / X_train["smoker"].std()) * X_train["age"].std()
print("\nAfter standardization, a smoker vs non-smoker difference weighs as much as an age gap of (years): ",
      round(smoker_in_age_units, 3))

Standardizing turns kNN from slightly better than the baseline into the best model so far.

- **Unscaled:** R^2 = 0.195, MAE \$7,682.710, RMSE \$10,927.458, slightly better than always predicting the mean.
- **Standardized:** R^2 = 0.771, MAE \$3,468.019, RMSE \$5,830.871, better than linear regression.

**Reason**: kNN predicts by averaging the charges of the nearest training people, and Euclidean distance adds up squared differences feature by feature. Without scaling, a feature's influence on the distance is proportional to its numeric spread.

`age` has a standard deviation of about 14 years and `bmi` about 6, while `smoker` is a 0/1 column with a standard deviation of 0.4: the largest possible smoker difference (1) counts less than a single year of age. So the unscaled model picks neighbors almost only by age and BMI and almost ignores smoking which is the strongest driver of cost. The neighbor check confirms it. For test smokers, only **30.2%** of their unscaled neighbors are smokers, so a smoker's prediction is mostly an average of non-smokers' much lower charges.

After standardization, every feature is measured in standard deviations of the training data. A smoker vs non-smoker difference (1 / 0.4 ~ 2.5 standard deviations) now weighs same as an age gap of about 35 years, and 99.9% of the neighbors share the test person's smoker status. Linear regression was also scaled (Section 6), but for it scaling does not change the predictions, because its coefficients adjust to each feature's units.

#### Visualizing the Effect of Scaling on kNN - Task c.i

This cell shows the k = 5 comparison above in four panels:

1. **Test-set MAE and RMSE** for unscaled vs standardized kNN, with R² under each model. Dashed lines mark the baseline and linear-regression RMSE from Section 6 for reference.
2. **Feature spread before vs after standardization:** the standard deviation of each training feature. Before scaling, this decides how much each feature can contribute to a Euclidean distance.
3. **Actual vs predicted charges, unscaled kNN**, colored by smoker status. The title shows the share of test smokers' 5 neighbors who are also smokers.
4. **The same plot for standardized kNN**, on the same axes so the two are directly comparable.

It also prints the mean actual and mean predicted charges for test smokers under both versions.

In [ ]:
# Reference RMSE lines from Section 6
baseline_test_rmse = results_b.loc[results_b["Model"] == "Baseline (train mean)", "RMSE (USD)"].iloc[0]
lr_test_rmse = results_b.loc[results_b["Model"] == "Linear Regression", "RMSE (USD)"].iloc[0]

model_names_c1 = ["kNN k=5\n(unscaled)", "kNN k=5\n(standardized)"]
x_pos = np.arange(len(model_names_c1))
bar_width = 0.30

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# (1) Test-set MAE and RMSE, with R² under each model name and reference lines
mae_bars = axes[0, 0].bar(x_pos - bar_width / 2, results_c1["MAE (USD)"], bar_width,
                          color="blue", edgecolor="black", label="MAE")
rmse_bars = axes[0, 0].bar(x_pos + bar_width / 2, results_c1["RMSE (USD)"], bar_width,
                           color="orange", edgecolor="black", label="RMSE")
axes[0, 0].bar_label(mae_bars, fmt="{:,.0f}", padding=3)
axes[0, 0].bar_label(rmse_bars, fmt="{:,.0f}", padding=3)
axes[0, 0].axhline(baseline_test_rmse, color="gray", linestyle="--", linewidth=1.5,
                   label=f"Baseline RMSE ({baseline_test_rmse:,.0f})")
axes[0, 0].axhline(lr_test_rmse, color="purple", linestyle="--", linewidth=1.5,
                   label=f"Linear Regression RMSE ({lr_test_rmse:,.0f})")
axes[0, 0].set_xticks(x_pos)
axes[0, 0].set_xticklabels([f"{name}\nR² = {r2:.3f}" for name, r2 in zip(model_names_c1, results_c1["R2"])])
axes[0, 0].set_title("Test-set Errors: kNN Unscaled vs Standardized")
axes[0, 0].set_ylabel("Error (USD)")
axes[0, 0].set_ylim(0, baseline_test_rmse * 1.35)
axes[0, 0].legend(loc="upper right")
axes[0, 0].grid(True, axis="y", alpha=0.3)

# (2) Spread of each feature before and after standardization (training set)
feature_names = X_train.columns.tolist()
std_unscaled = X_train.std()
std_scaled = pd.DataFrame(scaler_fit.transform(X_train), columns=feature_names).std()
f_pos = np.arange(len(feature_names))

unscaled_bars = axes[0, 1].bar(f_pos - bar_width / 2, std_unscaled, bar_width,
                               color="gray", edgecolor="black", label="Unscaled")
axes[0, 1].bar(f_pos + bar_width / 2, std_scaled, bar_width,
               color="teal", edgecolor="black", label="Standardized")
axes[0, 1].bar_label(unscaled_bars, fmt="{:.1f}", padding=3)
axes[0, 1].set_xticks(f_pos)
axes[0, 1].set_xticklabels(feature_names, rotation=35, ha="right")
axes[0, 1].set_title("Feature Spread Before vs After Standardization (training set)")
axes[0, 1].set_ylabel("Standard deviation (feature units)")
axes[0, 1].legend()
axes[0, 1].grid(True, axis="y", alpha=0.3)

# (3) and (4) Actual vs predicted charges, colored by smoker status
low = -2000
high = max(y_test.max(), y_pred_knn_raw.max(), y_pred_knn_scaled.max()) + 2000

scatter_panels = [
    (axes[1, 0], y_pred_knn_raw, "unscaled", match_raw[is_smoker].mean()),
    (axes[1, 1], y_pred_knn_scaled, "standardized", match_scaled[is_smoker].mean()),
]

for ax, y_pred, version, smoker_share in scatter_panels:
    for value, label, color in smoker_groups:
        mask = X_test["smoker"].values == value
        ax.scatter(y_test.values[mask], y_pred[mask], s=18, alpha=0.6, color=color, label=label)
    ax.plot([low, high], [low, high], color="black", linestyle="--", linewidth=1,
            label="Perfect prediction (y = x)")
    ax.set_xlim(low, high)
    ax.set_ylim(low, high)
    ax.set_title(f"Actual vs Predicted, kNN k=5 ({version})\n"
                 f"{smoker_share:.1%} of test smokers' 5 neighbors are smokers")
    ax.set_xlabel("Actual charges (USD)")
    ax.set_ylabel("Predicted charges (USD)")
    ax.legend(loc="upper left")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("knn_unscaled_vs_standardized.png", dpi=250)
plt.show()

smoker_actual_mean = y_test.values[is_smoker].mean()
smoker_pred_raw_mean = y_pred_knn_raw[is_smoker].mean()
smoker_pred_scaled_mean = y_pred_knn_scaled[is_smoker].mean()

print("\nTest smokers (n =", int(is_smoker.sum()), "):")
print("  Mean actual charges (USD): ", round(smoker_actual_mean, 3))
print("  Mean predicted, unscaled (USD): ", round(smoker_pred_raw_mean, 3))
print("  Mean predicted, standardized (USD): ", round(smoker_pred_scaled_mean, 3))

**What the plots show:**

- **Top left:** unscaled kNN (RMSE \$10,927.458) sits not far under the baseline line (\$12,179.404), so it improved little. Standardized kNN (RMSE \$5,830.871) drops below the linear-regression line (\$6,238.145). The only difference between the two models is scaling.
- **Top right:** this is the cause. Before scaling, `age` (std 14.0) and `bmi` (std 6.2) are far larger than every other feature, while `smoker` has a std of only 0.4. Euclidean distance is therefore decided almost entirely by age and BMI. After standardization, every feature has a std of about 1, so each one has a fair say in the distance.
- **Bottom left (unscaled):** errors go in both directions.
    - Smokers (red) are all below the y = x line, most of them far below. Their neighbors are mostly non-smokers of similar age and BMI (only 30.2% are smokers), so they get non-smoker-level predictions. Their mean predicted charge is \$14,845.072 vs an actual mean of \$31,910.784.
    - Non-smokers spread above the line, some predicted near \$30,000, because their similar-age, similar-BMI neighbors include smokers.
- **Bottom right (standardized):** 99.6% of smokers' neighbors are smokers, and both groups follow the y = x line. The smokers' mean prediction (\$32,960.852) is now close to their actual mean. The main remaining misses are non-smokers below the line at actual charges of about \$11,000–\$30,000. This is the unexplained middle band.

### Section 9: kNN Hyperparameter Optimization with 5-Fold CV - Task c.ii

This cell builds a `Pipeline` (StandardScaler → kNN) and searches over:

- `n_neighbors` (k) = {1, 2, …, 30},
- `weights` = {uniform, distance}: *uniform* is a simple average of the k neighbors; *distance* gives closer neighbors a larger weight (1 / distance),
- `p` = {1, 2}: the distance metric, L1/Manhattan (p = 1) vs L2/Euclidean (p = 2).

That is 30 × 2 × 2 = 120 combinations, each scored with 5-fold cross-validation on the training data only. Because the scaler is a pipeline step, it is re-fit inside each training fold, so no information from the validation fold leaks into the scaling. The training rows were already shuffled by `train_test_split` with the fixed seed, so the 5 folds are reproducible.

Scoring uses `neg_root_mean_squared_error`. `return_train_score=True` also keeps the training-fold RMSE, which we use later. The `knn__` prefix tells the pipeline that the parameter belongs to the step named `"knn"`.

In [ ]:
knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsRegressor())
])

param_grid_knn = {
    "knn__n_neighbors": list(range(1, 31)),
    "knn__weights": ["uniform", "distance"],
    "knn__p": [1, 2]
}

grid_search_knn = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=param_grid_knn,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1,
    return_train_score=True
)

grid_search_knn.fit(X_train, y_train)

best_k = grid_search_knn.best_params_["knn__n_neighbors"]
best_weights = grid_search_knn.best_params_["knn__weights"]
best_p = grid_search_knn.best_params_["knn__p"]
best_cv_rmse = -grid_search_knn.best_score_
best_cv_std = grid_search_knn.cv_results_["std_test_score"][grid_search_knn.best_index_]
best_metric_name = "Manhattan / L1" if best_p == 1 else "Euclidean / L2"

print("\nCombinations searched: ", len(grid_search_knn.cv_results_["params"]))
print("\nBest k (n_neighbors): ", best_k)
print("\nBest weights: ", best_weights)
print("\nBest p: ", best_p, "(" + best_metric_name + ")")
print("\nBest 5-fold CV RMSE (USD): ", round(best_cv_rmse, 3))
print("\nStd of RMSE across the 5 folds (USD): ", round(best_cv_std, 3))

`GridSearchCV` has already re-fit the best pipeline on the full training set (`best_estimator_`). This cell evaluates that tuned model on the held-out test set. For reference, it also shows the default k = 5 standardized model from Section 8, together with its CV RMSE from the same search, so we can compare how CV and the test set rank the two.

In [ ]:
knn_tuned = grid_search_knn.best_estimator_
y_pred_knn_tuned = knn_tuned.predict(X_test)

tuned_label = f"kNN tuned (k={best_k}, {best_weights}, p={best_p})"

results_c2 = pd.DataFrame([
    evaluate(tuned_label, y_test, y_pred_knn_tuned),
    evaluate("kNN k=5 (uniform, p=2), untuned", y_test, y_pred_knn_scaled),
])

# Tidy table of the full search (used again in Task c.iii)
cv_results_knn = pd.DataFrame(grid_search_knn.cv_results_)
cv_results_knn["k"] = cv_results_knn["param_knn__n_neighbors"].astype(int)
cv_results_knn["weights"] = cv_results_knn["param_knn__weights"].astype(str)
cv_results_knn["p"] = cv_results_knn["param_knn__p"].astype(int)
cv_results_knn["cv_rmse"] = -cv_results_knn["mean_test_score"]
cv_results_knn["train_rmse"] = -cv_results_knn["mean_train_score"]

default_row = cv_results_knn[(cv_results_knn["k"] == 5) &
                             (cv_results_knn["weights"] == "uniform") &
                             (cv_results_knn["p"] == 2)]
default_cv_rmse = default_row["cv_rmse"].iloc[0]

print("\nTest-set metrics:")
print(results_c2.to_string(index=False))
print("\n5-fold CV RMSE, tuned model (USD): ", round(best_cv_rmse, 3))
print("\n5-fold CV RMSE, default k=5 model (USD): ", round(default_cv_rmse, 3))

The best setting is **k = 8, distance weighting, Euclidean (L2) distance**, with a 5-fold CV RMSE of **\$5,555.633**. On the test set the tuned kNN reaches **R^2 = 0.786, MAE = \$3,283.485 and RMSE = \$5,639.828**, which is better than linear regression (RMSE \$6,238.145).

The untuned k = 5 model is worse on both measures: its CV RMSE is \$5,677.523 vs \$5,555.633, and its test RMSE is \$5,830.871 vs \$5,639.828, so cross-validation and the test set agree here. The test-set difference of about \$191 is still inside the fold-to-fold variation (standard deviation ~ \$299). Choosing _k_ by its test score would turn the test set into a tuning set causing leakage and make the reported test error optimistic. Cross-validation on the training data is the principled way to choose, and the test set stays an unbiased final check.

### Section 10: Cross-Validated RMSE vs k (Bias–Variance) - Task c.iii

The left panel plots the 5-fold CV RMSE against _k_ for all four weights/distance combinations, with the best _k_ marked. The right panel takes uniform weights with the best distance metric and adds the mean training-fold RMSE next to the CV RMSE, so under and overfitting are visible directly. Uniform weights are used there because with distance weights each training point is its own neighbor at distance 0 and zero-distance neighbors dominate the prediction, so the training error is close to 0 and tells us nothing. The cell also prints the values at the extremes of k.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

line_styles = {
    ("uniform", 1): ("blue", "--"),
    ("uniform", 2): ("blue", "-"),
    ("distance", 1): ("orange", "--"),
    ("distance", 2): ("orange", "-"),
}

for (w, p), (color, ls) in line_styles.items():
    sub = cv_results_knn[(cv_results_knn["weights"] == w) & (cv_results_knn["p"] == p)].sort_values("k")
    dist_name = "Manhattan" if p == 1 else "Euclidean"
    axes[0].plot(sub["k"], sub["cv_rmse"], color=color, linestyle=ls, marker="o", markersize=3,
                 label=f"{w} weights, {dist_name}")

axes[0].axvline(best_k, color="gray", linestyle=":", label=f"best k = {best_k}")
axes[0].set_title("5-fold CV RMSE vs k (all searched settings)")
axes[0].set_xlabel("Number of neighbors (k)")
axes[0].set_ylabel("5-fold CV RMSE (USD)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

uniform_best_p = cv_results_knn[(cv_results_knn["weights"] == "uniform") &
                                (cv_results_knn["p"] == best_p)].sort_values("k")
dist_name_best = "Manhattan" if best_p == 1 else "Euclidean"

axes[1].plot(uniform_best_p["k"], uniform_best_p["train_rmse"], color="teal", marker="o", markersize=3,
             label="Training-fold RMSE")
axes[1].plot(uniform_best_p["k"], uniform_best_p["cv_rmse"], color="purple", marker="o", markersize=3,
             label="Validation-fold (CV) RMSE")
axes[1].set_title(f"Training vs CV RMSE (uniform weights, {dist_name_best})")
axes[1].set_xlabel("Number of neighbors (k)")
axes[1].set_ylabel("RMSE (USD)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("knn_cv_rmse_vs_k.png", dpi=250)
plt.show()

best_curve = cv_results_knn[(cv_results_knn["weights"] == best_weights) &
                            (cv_results_knn["p"] == best_p)].set_index("k")

print("\nCV RMSE for the best setting (" + best_weights + ", " + dist_name_best + ") (USD):")
for k_value in [1, best_k, 30]:
    print("  k =", k_value, ":", round(best_curve.loc[k_value, "cv_rmse"], 3))

print("\nUniform weights, " + dist_name_best + ": training-fold vs CV RMSE (USD):")
for k_value in [1, best_k, 30]:
    row = uniform_best_p.set_index("k").loc[k_value]
    print("  k =", k_value, ": train =", round(row["train_rmse"], 3) + 0.0, "| CV =", round(row["cv_rmse"], 3),
          "| gap =", round(row["cv_rmse"] - row["train_rmse"], 3))

euclid = cv_results_knn[cv_results_knn["p"] == 2].set_index(["weights", "k"])["cv_rmse"]
manhat = cv_results_knn[cv_results_knn["p"] == 1].set_index(["weights", "k"])["cv_rmse"]

**Very small k (k = 1, 2): overfitting, high variance.**\
With k = 1 every prediction copies one training person's charge. The training error is low, yet the CV RMSE is the worst on the curve. The model memorizes noise: a slightly different training sample would give very different predictions. The large gap between the training and CV curves in the right panel is the signature of high variance.

**Very large k (toward 30 and beyond): underfitting, high bias.** Each prediction averages more and more people who are not really similar, so predictions are pulled toward the overall mean and local structure is smoothed away. Training and CV errors both rise and the gap between them shrinks, which is the signature of high bias. In the limit k = n every prediction equals the training mean, which is exactly the naive baseline.

**In between (k ≈ 7–11)** the CV curve reaches its minimum: neighborhoods are large enough to average out noise but small enough to keep local patterns. This is the bias–variance trade-off. _k_ plays the role that alpha plays for Lasso. Small k = more complex model, large k = simpler model.

Two more patterns: distance weighting beats uniform weighting for k >= 5 and the advantage grows with _k_. Because far-away neighbors get less weight, which slows the drift toward the mean. And Euclidean distance beats Manhattan at every k in this search except k = 1, 2.

### Section 11: Lasso Regression with Tuned alpha

Lasso is linear regression with an L1 penalty (alpha × the sum of the absolute coefficients) added to the squared-error loss. A larger alpha shrinks the coefficients more and can set some exactly to 0, which acts as feature selection. Because the penalty treats all coefficients the same, the features must be on the same scale, so Lasso sits in a `Pipeline` after a `StandardScaler`.

This cell tunes alpha with 5-fold CV on the training data, prints the CV RMSE for every alpha, reports the best alpha, and evaluates the tuned model on the test set. The alpha grid reaches into the thousands because the target is in dollars, so small alphas barely change the fit. `max_iter` is raised so the solver converges for every alpha.

In [ ]:
lasso_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("lasso", Lasso(max_iter=10000))
])

param_grid_lasso = {
    "lasso__alpha": [0.01, 0.1, 1, 10, 50, 100, 200, 500, 1000, 2000]
}

grid_search_lasso = GridSearchCV(
    estimator=lasso_pipeline,
    param_grid=param_grid_lasso,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1
)

grid_search_lasso.fit(X_train, y_train)

best_alpha = grid_search_lasso.best_params_["lasso__alpha"]
best_cv_rmse_lasso = -grid_search_lasso.best_score_

lasso_cv_table = pd.DataFrame({
    "alpha": grid_search_lasso.cv_results_["param_lasso__alpha"].astype(float),
    "CV RMSE (USD)": (-grid_search_lasso.cv_results_["mean_test_score"]).round(3),
})

print("\n5-fold CV RMSE for each alpha:")
print(lasso_cv_table.to_string(index=False))
print("\nBest alpha: ", best_alpha)
print("\nBest 5-fold CV RMSE (USD): ", round(best_cv_rmse_lasso, 3))

# Plot: 5-fold CV RMSE for every alpha in the grid (alphas evenly spaced so all are readable)
alpha_pos = np.arange(len(lasso_cv_table))
alpha_labels = [f"{a:g}" for a in lasso_cv_table["alpha"]]
is_best_alpha = (lasso_cv_table["alpha"] == best_alpha).values

plt.figure(figsize=(11, 5))
plt.plot(alpha_pos, lasso_cv_table["CV RMSE (USD)"], color="steelblue", marker="o", markersize=7,
         label="5-fold CV RMSE")
plt.scatter(alpha_pos[is_best_alpha], lasso_cv_table["CV RMSE (USD)"][is_best_alpha], s=150,
            color="darkorange", edgecolor="black", zorder=3, label=f"Best alpha = {best_alpha:g}")
for x, rmse in zip(alpha_pos, lasso_cv_table["CV RMSE (USD)"]):
    plt.annotate(f"{rmse:,.0f}", (x, rmse), textcoords="offset points", xytext=(0, 10),
                 ha="center", fontsize=9)
plt.xticks(alpha_pos, alpha_labels)
plt.title("Lasso: 5-fold CV RMSE for Each alpha (training data only)")
plt.xlabel("alpha (regularization strength)")
plt.ylabel("5-fold CV RMSE (USD)")
plt.ylim(lasso_cv_table["CV RMSE (USD)"].min() - 100, lasso_cv_table["CV RMSE (USD)"].max() + 150)
plt.legend(loc="upper left")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("lasso_cv_rmse_vs_alpha.png", dpi=150)
plt.show()

lasso_tuned = grid_search_lasso.best_estimator_
y_pred_lasso = lasso_tuned.predict(X_test)
lasso_label = f"Lasso (alpha={best_alpha})"

results_lasso = pd.DataFrame([
    evaluate(lasso_label, y_test, y_pred_lasso),
    evaluate("Linear Regression", y_test, y_pred_lr),
])

print("\nTest-set metrics:")
print(results_lasso.to_string(index=False))

This cell compares the Lasso coefficients with the plain linear-regression coefficients from Section 7. Lasso was fit on standardized features, so its raw coefficients are in USD per one standard deviation of each feature. Dividing by the training standard deviation stored in the scaler (`scale_`) converts them back to USD per original unit, which makes them directly comparable with linear regression.

In [ ]:
lasso_model = lasso_tuned.named_steps["lasso"]
lasso_scaler = lasso_tuned.named_steps["scaler"]

coef_compare = pd.DataFrame({
    "LR (USD per unit)": coef_lr,
    "Lasso (USD per 1 std)": pd.Series(lasso_model.coef_, index=X_train.columns),
    "Lasso (USD per unit)": pd.Series(lasso_model.coef_ / lasso_scaler.scale_, index=X_train.columns),
}).round(3) + 0.0   # "+ 0.0" turns -0.0 into 0.0 for display

zeroed = X_train.columns[lasso_model.coef_ == 0].tolist()

print(coef_compare.to_string())
print("\nFeatures set exactly to 0 by Lasso: ", zeroed)

Tuning picks alpha = 50, but the CV curve is essentially flat from alpha = 0.01 up to 100 (all around \$6,024). It only gets worse once alpha is large (>= 200) and starts shrinking useful coefficients. On the test set Lasso scores R^2 = 0.738, MAE = \$4,258.952 and RMSE = \$6,238.857, essentially the same as plain linear regression.

**Reason**: Regularization helps when a model overfits, but here linear regression has only 9 features, 1,069 training rows and similar training and test errors, so there is little variance for the penalty to remove. The CV RMSE at alpha = 0.01 also shows that tuned kNN (CV RMSE \$5,555.633) beats the linear models on the training folds too, not only on the test set.

Lasso sets `region_southwest` exactly to 0. Zeroing one region coefficient resolves the redundancy of the four one-hot columns - the remaining three region coefficients are now read relative to the southwest. The interpretation of `smoker` and `bmi` does not change.

### Section 12: Model Comparison Table - Task d.i

This cell gathers the test-set metrics of the naive baseline, linear regression and tuned kNN into one table, all computed on the same 268 held-out rows. The Lasso is added as a last row for reference.

In [ ]:
comparison = pd.DataFrame([
    evaluate("Baseline (train mean)", y_test, y_pred_baseline),
    evaluate("Linear Regression", y_test, y_pred_lr),
    evaluate(tuned_label, y_test, y_pred_knn_tuned),
    evaluate(lasso_label, y_test, y_pred_lasso),
])

print("\nTest-set metrics (268 held-out rows):")
print(comparison.to_string(index=False))

**Comparison table:** the tuned kNN has the best test metrics on all three measures (R^2 = 0.786, MAE = \$3,283.485, RMSE = \$5,639.828). Compared with linear regression (R² = 0.738, MAE = \$4,267.839, RMSE = \$6,238.145), its MAE is about \$984 (23%) lower and its RMSE about \$598 (10%) lower. Both models are far better than the naive baseline (R² = −0.000, RMSE = \$12,179.404), and the Lasso is almost identical to linear regression.

### Section 13: Residual Plots - Task d.i

This cell draws a residual plot for each model: residual = actual − predicted charges, plotted against the predicted value, with a dashed line at 0. Points above 0 are under-predictions, points below are over-predictions. A good model's residuals scatter evenly around 0 with no visible pattern. Points are colored by smoker status (same colors as Section 4), and all panels share both axes so error sizes are directly comparable. The cell also counts how many test predictions are negative and how many errors exceed \$5,000, \$10,000 and \$20,000.

In [ ]:
residual_models = [
    ("Baseline (train mean)", y_pred_baseline),
    ("Linear Regression", y_pred_lr),
    (tuned_label, y_pred_knn_tuned),
    (lasso_label, y_pred_lasso),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True, sharey=True)

for ax, (name, y_pred) in zip(axes.ravel(), residual_models):
    residuals = y_test.values - y_pred
    for value, label, color in smoker_groups:
        mask = X_test["smoker"].values == value
        ax.scatter(y_pred[mask], residuals[mask], alpha=0.6, s=18, color=color, label=label)
    ax.axhline(0, color="black", linestyle="--", linewidth=1)
    ax.set_title("Residuals vs Predicted: " + name)
    ax.set_xlabel("Predicted charges (USD)")
    ax.set_ylabel("Residual = actual − predicted (USD)")
    ax.legend(loc="upper right")
    ax.grid(True, alpha=0.3)
    ax.tick_params(labelbottom=True, labelleft=True)

plt.tight_layout()
plt.savefig("residual_plots.png", dpi=150)
plt.show()

error_counts = pd.DataFrame([
    {
        "Model": name,
        "Negative predictions": int((y_pred < 0).sum()),
        "|residual| > 5k": int((np.abs(y_test.values - y_pred) > 5000).sum()),
        "|residual| > 10k": int((np.abs(y_test.values - y_pred) > 10000).sum()),
        "|residual| > 20k": int((np.abs(y_test.values - y_pred) > 20000).sum()),
    }
    for name, y_pred in residual_models
])

print("\nCounts out of 268 test rows:")
print(error_counts.to_string(index=False))
print("\nLowest Linear Regression prediction (USD): ", round(y_pred_lr.min(), 3))

**What the residual plots show:**

- *Baseline:* every prediction is the same (\$13,275), so all points sit on one vertical line. Smokers are almost all above 0 (under-predicted) and non-smokers below 0 (over-predicted): the residuals are simply all the variation the baseline ignores.
- *Linear regression and Lasso:* non-smokers form a tight cloud near 0, but smokers split into two parallel bands, one about \$5,000–\$9,500 above 0 and one about \$7,000–\$11,000 below 0, with no smoker residuals near 0. There is also a gap where the model never predicts (roughly \$18,000–\$24,000), because the `smoker` coefficient shifts every smoker's prediction up by the same \$23,899. Two predictions are negative, an impossible cost.
- *Tuned kNN:* smoker residuals are scattered around 0 instead of split into two bands, and most non-smokers are close to 0. The errors are fewer: kNN has fewer errors above \$5,000 (57 vs 82), above \$10,000 (23 vs 30) and above \$20,000 (3 vs 5). There are no negative predictions, because kNN averages real charges.
- *All models:* a group of non-smokers sits about \$8,000–\$22,000 above 0 in every panel. These are the middle-band non-smokers whose high costs are not explained by any of the six features, so no choice of model fixes them.

### Section 14: Checking the Residual Patterns by Group - Task d.ii

The residual plots suggest the errors are not random. To check this with numbers, this cell compares the **mean residual** across the four groups formed by smoker status and BMI below vs at/above 30 (the same groups as Section 4). A mean residual far from 0 means a model systematically under-predicts (positive) or over-predicts (negative) that group, which a single overall metric cannot show.

In [ ]:
residual_groups = pd.DataFrame({
    "smoker": X_test["smoker"].map({0: "Non-smoker", 1: "Smoker"}).values,
    "bmi_group": np.where(X_test["bmi"].values >= 30, "BMI >= 30", "BMI < 30"),
    "LR residual": y_test.values - y_pred_lr,
    "kNN residual": y_test.values - y_pred_knn_tuned,
})

group_residuals = residual_groups.groupby(["smoker", "bmi_group"]).agg(
    count=("LR residual", "size"),
    mean_LR_residual=("LR residual", "mean"),
    mean_kNN_residual=("kNN residual", "mean"),
).round(3)

print("\nMean residual (actual − predicted, USD) by group, test set:")
print(group_residuals.to_string())

The group check confirms the plots. Linear regression under-predicts smokers with BMI >= 30 by \$9,243.330 on average and over-predicts smokers with BMI < 30 by \$9,070.000. For non-smokers the signs reverse and the errors are smaller (+\$678.083 below BMI 30, −\$1,021.923 at or above). This is exactly what we predicted - one BMI slope (\$334 per kg/m^2) is too flat for smokers and too steep for non-smokers. kNN reduces these systematic errors substantially for smokers (+\$5,009.022 and −\$5,547.657) and makes them smaller for non-smokers (−\$525.052 and +\$590.274), because it does not force one slope on everyone.

**Model to choose**: The tuned kNN (k = 8, distance weights, Euclidean distance), for predicting charges given this data.

**Reasons**:
- *Accuracy:* it is the most accurate model on the test set on all three metrics. Its CV RMSE on the training data also beats the linear models, and it was chosen without looking at the test set.
- *Residuals:* its errors are less systematic. Linear regression is wrong in a predictable way for the high BMI smokers group, the most expensive people in the data, and it can produce impossible negative costs.

**The accuracy vs interpretability trade-off**: Linear regression's main strength is interpretability: each coefficient is a dollar amount per unit of a feature, which is easy to explain. kNN has no coefficients and a prediction can only be explained by showing the similarity between people it averaged. It must store and search every training row at prediction time, and it depends on correct scaling. If the goal were only to explain charges rather than predict them linear regression would be the more natural tool.

However, the residual plots show that the linear model's simple story is misleading on this data: "BMI adds \$334 per year" is not accurate for either smokers or non-smokers, and the model under-predicts high BMI smokers by about \$9,000. An interpretable model is only useful if its interpretation is correct, so the linear model's form would first need to account for how smoking and BMI combine before its coefficients could be trusted.

**What the residual plots reveal that the metrics alone do not:**

1. *Where the errors are.* R^2 = 0.738 looks good, but the plot shows that linear regression's errors are concentrated in one group (smokers) and follow a clear pattern, which signals a wrong model form rather than random noise.
2. *Impossible predictions.* A negative charge is invisible in R^2, MAE or RMSE.
3. *The type of error.* kNN's lower RMSE comes with errors that are spread around 0 instead of the two bands linear regression shows for smokers, but kNN still misses some individual smokers by \$10,000 or more, the variance side of the trade-off seen in Task c.iii.
4. *What no model can fix.* The cluster of under-predicted non-smokers in every panel shows that part of the remaining error comes from information that is not in the data, not from the choice of model.